In [0]:
import yaml

with open("../config/config.yaml") as f:
    config = yaml.safe_load(f)

catalog, schema = config["catalog"], config["schema"]
colecoes = [c["collection"] for c in config["collections"]]
control_table = config["control_table"]

print("=== Validação de entrega ===\n")

1. Pipeline processa todas as coleções

In [0]:
faltando = [c for c in colecoes if not spark.catalog.tableExists(f"{catalog}.{schema}.{c}")]
if faltando:
    print(f"FALHOU: coleções sem tabela Bronze: {faltando}")
else:
    print("OK: todas as coleções têm tabela Bronze")

2. Carga incremental com watermark persistida

In [0]:
incrementais = [c["collection"] for c in config["collections"] if c["modo_carga"] == "incremental"]
df_log = spark.table(control_table)

for c in incrementais:
    tem_watermark = df_log.filter((df_log.collection == c) & df_log.watermark_final.isNotNull()).count() > 0
    print(f"{'OK' if tem_watermark else 'FALHOU'}: watermark persistida para '{c}'")

3. control_ingestion_log com pelo menos 3 execuções

In [0]:
for c in colecoes:
    qtd_execucoes = df_log.filter(df_log.collection == c).count()
    status = "OK" if qtd_execucoes >= 3 else f"FALHOU (só {qtd_execucoes})"
    print(f"{status}: execuções registradas para '{c}'")

4. Colunas de rastreabilidade (R4) em todas as tabelas Bronze

In [0]:
colunas_obrigatorias = {"_ingestion_id", "_ingestion_timestamp", "_source_path", "_load_type", "_ingestion_date"}

for c in colecoes:
    colunas = {f.name for f in spark.table(f"{catalog}.{schema}.{c}").schema}
    faltando = colunas_obrigatorias - colunas
    print(f"{'OK' if not faltando else f'FALHOU: faltam {faltando}'}: colunas de controle em '{c}'")

5. Reconciliação: contagem origem × destino bate, sem duplicados

In [0]:
for c in colecoes:
    ultima = df_log.filter(df_log.collection == c).orderBy(df_log.start_time.desc()).first()
    if ultima is None:
        continue
    bateu = ultima.qtd_lida_origem == ultima.qtd_gravada_destino
    print(f"{'OK' if bateu else 'FALHOU'}: contagem origem x destino em '{c}' "
          f"(lida={ultima.qtd_lida_origem}, gravada={ultima.qtd_gravada_destino})")

    dup = (spark.table(f"{catalog}.{schema}.{c}")
             .groupBy("_source_id", "_ingestion_date").count()
             .filter("count > 1").count())
    print(f"{'OK' if dup == 0 else f'FALHOU: {dup} duplicados'}: idempotência em '{c}'")

6. Coluna de quarentena (R7) existe

In [0]:
for c in colecoes:
    colunas = {f.name for f in spark.table(f"{catalog}.{schema}.{c}").schema}
    print(f"{'OK' if '_rescued_data' in colunas else 'FALHOU'}: coluna _rescued_data em '{c}'")